In [5]:

!pip install datasets==2.19.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [1]:
from datasets import load_dataset

print("Downloading conll2003...")
# Because we downgraded, trust_remote_code=True will work again!
dataset = load_dataset("conll2003", trust_remote_code=True)

print("\nDownload Complete! Here is the dataset structure:")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]


Download Complete! Here is the dataset structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [3]:
!pip install -q transformers datasets evaluate seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [6]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
import evaluate

# ==========================================
# Setup Device
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# ==========================================
# TASK 1: Dataset Selection
# ==========================================
print("--- TASK 1: Dataset Selection ---")
# Added trust_remote_code=True to prevent Colab security blocks
raw_datasets = load_dataset("conll2003", trust_remote_code=True)

chunk_features = raw_datasets["train"].features["chunk_tags"]
label_names = chunk_features.feature.names

print("Dataset Name: CoNLL-2003")
print(f"Label Categories (Chunk Tags): {label_names}\n")

# ==========================================
# TASK 2: Data Preprocessing
# ==========================================
print("--- TASK 2: Data Preprocessing ---")
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
print("Preprocessing complete.\n")

# ==========================================
# TASK 3: Model Setup
# ==========================================
print("--- TASK 3: Model Setup ---")
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
).to(device)

print(f"Model configured with {model.config.num_labels} labels.\n")

# ==========================================
# Evaluation Metrics Setup
# ==========================================
metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

# ==========================================
# TASK 4: Training
# ==========================================
print("--- TASK 4: Training ---")
small_train_dataset = tokenized_datasets["train"].select(range(2000))
small_eval_dataset = tokenized_datasets["validation"].select(range(500))

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="./distilbert-chunking",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer, # <--- THIS IS THE UPDATED LINE
)

print("Starting Training...")
trainer.train()

# ==========================================
# TASK 5: Final Evaluation
# ==========================================
print("\n--- TASK 5: Final Evaluation ---")
eval_results = trainer.evaluate()
print(eval_results)

# ==========================================
# TASK 6: Inference
# ==========================================
print("\n--- TASK 6: Inference ---")
text = "John works at Google in California."

chunking_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

predictions = chunking_pipeline(text)

print(f"Input: {text}")
print("Predicted Chunk Tags:")
for pred in predictions:
    print(f"Word: {pred['word']:<10} | Tag: {pred['entity_group']:<5} | Confidence: {pred['score']:.4f}")

Using device: cpu

--- TASK 1: Dataset Selection ---
Dataset Name: CoNLL-2003
Label Categories (Chunk Tags): ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']

--- TASK 2: Data Preprocessing ---


Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Preprocessing complete.

--- TASK 3: Model Setup ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.646979,0.394077,0.818921,0.817109,0.818014,0.907368
2,0.316747,0.322114,0.845723,0.834808,0.840230,0.921053
3,0.256147,0.310790,0.860150,0.843658,0.851824,0.925614


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



--- TASK 5: Final Evaluation ---


{'eval_loss': 0.310790479183197, 'eval_precision': 0.8601503759398497, 'eval_recall': 0.8436578171091446, 'eval_f1': 0.851824274013403, 'eval_accuracy': 0.9256140350877193, 'eval_runtime': 27.835, 'eval_samples_per_second': 17.963, 'eval_steps_per_second': 1.15, 'epoch': 3.0}

--- TASK 6: Inference ---
Input: John works at Google in California.
Predicted Chunk Tags:
Word: john       | Tag: NP    | Confidence: 0.9753
Word: works      | Tag: VP    | Confidence: 0.8689
Word: at         | Tag: PP    | Confidence: 0.9604
Word: google     | Tag: NP    | Confidence: 0.9581
Word: in         | Tag: PP    | Confidence: 0.9708
Word: california | Tag: NP    | Confidence: 0.9609


In [ ]:
Task 7: Comparison (POS Tagging vs. Chunking)
POS Tagging (Part-of-Speech): This is a grammar-level tagging task. It is generally considered easier because it assigns a grammatical category to every single word individually. It looks at the word and its immediate neighbors. (Example: "Google" -> Noun, "works" -> Verb).
Chunking (Phrase Detection): This is a phrase-level grouping task. It is considered medium difficulty because it groups consecutive words together into syntactic phrases (e.g., Noun Phrase, Verb Phrase, Prepositional Phrase). It requires understanding the broader structure of the sentence and heavily relies on the BIO format (Begin, Inside, Outside). (Example: "The big black dog" -> B-NP, I-NP, I-NP, I-NP).

Task 8: Report / Blog
1. Differences Between POS Tagging and Chunking:
As noted in Task 7, POS tagging identifies individual word types, whereas chunking identifies the boundaries of larger grammatical phrases. Chunking builds upon the foundation of POS tagging to understand how words interact as a single unit.
2. Challenges Faced:
The biggest challenge in token classification with Transformer models is Label Alignment. Models like BERT use subword tokenizers (WordPiece). A single word might be split into multiple pieces (e.g., "Washington" -> "Wash", "##ing", "##ton"). Since we only have one grammatical label for the original word, we must carefully align the labels by assigning the true label to the first subword and masking the remaining subwords with the -100 index so the loss function ignores them.
3. Observations and Insights:
Context Matters: Transformer models excel at Chunking because their self-attention mechanism looks at the entire sentence context simultaneously. This allows the model to easily figure out where a Noun Phrase begins and ends.
Transfer Learning is Powerful: Even training on a tiny subset of the CoNLL-2003 dataset (just 2,000 sentences) for only 3 epochs yielded an F1 score above 85%. This highlights the power of using a pre-trained base model like distilbert-base-uncased, which already possesses a deep foundational understanding of the English language.